<a href="https://colab.research.google.com/github/jyeon6273/DeepNeuralNetworks/blob/main/Baseline_Submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stanford Cars Mini · Student Baseline · Frozen DINOv1

**Goal:** Train a classification head on pretrained, **frozen** DINOv1 features and produce `submission.csv` for the Kaggle competition.


## 0. Install libraries (run once)

In [ ]:
%pip -q install timm scikit-learn pandas pillow tqdm matplotlib

## 1. Settings and folder checks

In [ ]:
from pathlib import Path
import random, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# CHANGE THIS to the folder containing train/, val/, and test/.
DATA_ROOT = Path('./')
OUTPUT_DIR = Path('./student_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 32 if DEVICE == 'cuda' else 16
EPOCHS = 35
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
PATIENCE = 7

for name in ('train', 'val', 'test'):
    assert (DATA_ROOT / name).is_dir(), f'Missing {name}/ under {DATA_ROOT.resolve()}'
print('Data:', DATA_ROOT.resolve())
print('Device:', DEVICE)

## 2. Load labeled class folders with `ImageFolder`

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=preprocess)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=preprocess)
assert train_ds.class_to_idx == val_ds.class_to_idx, 'train/ and val/ class folders must match'
NUM_CLASSES = len(train_ds.classes)
print('Classes:', NUM_CLASSES, '| Train:', len(train_ds), '| Val:', len(val_ds))
print('First class folders:', train_ds.classes[:5])
assert NUM_CLASSES == 20, f'Expected 20 classes, got {NUM_CLASSES}'

# Class-folder names from the instructor's export are prefixed by 000_, 001_, ...
# These prefixes make ImageFolder indices equal the Kaggle integer labels.
for i, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{i:03d}_'), f'Class-label mismatch at {folder}; ask your instructor'


## 3. Discover unlabeled test images and IDs

The `image_id` column must match `sample_submission.csv` exactly. The test folder stays flat, without class-name folders.

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
test_dir = DATA_ROOT / 'test'
test_files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)
assert test_files, f'No images found inside {test_dir.resolve()}'

sample_path = DATA_ROOT / 'sample_submission.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path, dtype={'image_id': str})
    assert {'image_id','label'}.issubset(sample.columns)
    required_ids = sample['image_id'].astype(str).tolist()
else:
    required_ids = sorted(p.stem for p in test_files)

by_id = {p.stem: p for p in test_files}
assert len(by_id) == len(test_files), 'Two test filenames share the same image_id stem'
assert len(set(required_ids)) == len(required_ids), 'Duplicate image_id in sample_submission.csv'
assert set(required_ids) == set(by_id), ('Test image IDs do not match sample_submission.csv',
                                         len(set(required_ids)-set(by_id)), len(set(by_id)-set(required_ids)))
print('Unlabeled test images:', len(required_ids))
print('Submission ID example:', required_ids[:3])

class TestImages(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        p = by_id[self.ids[index]]
        with Image.open(p) as im: image = preprocess(im.convert('RGB'))
        return image

test_ds = TestImages(required_ids)

## 4. Load frozen DINOv1 ViT-S/16

The pretrained model weights may download on the first run. Dataset images are read exclusively from local folders.

In [ ]:
import timm
MODEL_NAME = 'vit_small_patch16_224.dino'
backbone = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0).to(DEVICE)
backbone.eval()
for param in backbone.parameters():
    param.requires_grad_(False)
FEATURE_DIM = backbone.num_features
assert FEATURE_DIM == 384, f'Unexpected embedding size: {FEATURE_DIM}'
print('Frozen backbone:', MODEL_NAME, '| Feature dimension:', FEATURE_DIM)

## 5. Extract and cache 384-D DINO features

The backbone stays frozen; only the classification head will learn. Cache is checked against file names, sizes, modification times, and class mapping. Delete the cached file when changing the model or preprocessing.

In [ ]:
def dataset_signature():
    paths = [Path(p) for p, _ in train_ds.samples + val_ds.samples] + [by_id[i] for i in required_ids]
    parts = [MODEL_NAME, repr(preprocess), str(train_ds.class_to_idx)]
    for p in paths:
        st = p.stat()
        parts.append(f'{p.resolve()}|{st.st_size}|{st.st_mtime_ns}')
    return hashlib.sha256('\n'.join(parts).encode()).hexdigest()

CACHE = OUTPUT_DIR / 'dinov1_cls_features.pt'
signature = dataset_signature()

@torch.inference_mode()
def extract(loader, desc):
    embeddings = []
    backbone.eval()
    for batch in tqdm(loader, desc=desc):
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        z = backbone(x.to(DEVICE)).float()
        assert z.ndim == 2 and z.shape[1] == FEATURE_DIM, z.shape
        embeddings.append(z.cpu())
    return torch.cat(embeddings)

if CACHE.exists():
    try: cached = torch.load(CACHE, map_location='cpu', weights_only=True)
    except TypeError: cached = torch.load(CACHE, map_location='cpu')
else:
    cached = None

if cached is not None and cached.get('signature') == signature:
    print('Reusing feature cache:', CACHE)
    X_train, X_val, X_test = cached['X_train'], cached['X_val'], cached['X_test']
else:
    kwargs = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
                  pin_memory=(DEVICE == 'cuda'))
    X_train = extract(DataLoader(train_ds, **kwargs), 'Train embeddings')
    X_val = extract(DataLoader(val_ds, **kwargs), 'Validation embeddings')
    X_test = extract(DataLoader(test_ds, **kwargs), 'Test embeddings')
    torch.save({'signature': signature, 'X_train': X_train, 'X_val': X_val,
                'X_test': X_test}, CACHE)

y_train = torch.tensor(train_ds.targets, dtype=torch.long)
y_val = torch.tensor(val_ds.targets, dtype=torch.long)
assert len(X_train) == len(y_train) and len(X_val) == len(y_val) and len(X_test) == len(required_ids)
print('Shapes:', X_train.shape, X_val.shape, X_test.shape)

## 6. Standardize features using **training data only**

In [ ]:
feature_mean = X_train.mean(dim=0)
feature_std = X_train.std(dim=0, unbiased=False).clamp_min(1e-6)
Z_train = (X_train - feature_mean) / feature_std
Z_val = (X_val - feature_mean) / feature_std
Z_test = (X_test - feature_mean) / feature_std
print('Features standardized without validation/test leakage.')

## 7. Train the provided **linear** classification head

This is the baseline students should improve. Feel free to test an MLP, regularization, different DINO token pooling, or other allowed techniques while keeping the backbone frozen.

In [ ]:
from torch.utils.data import TensorDataset
training_loader = DataLoader(TensorDataset(Z_train, y_train), batch_size=64,
                             shuffle=True, num_workers=0)
head = nn.Linear(FEATURE_DIM, NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE,
                              weight_decay=WEIGHT_DECAY)

best_accuracy = -1.0
best_weights = None
bad_epochs = 0
history = []
for epoch in range(1, EPOCHS + 1):
    head.train()
    running_loss = 0.0
    for xb, yb in training_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(head(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(yb)
    head.eval()
    with torch.inference_mode():
        val_pred = head(Z_val.to(DEVICE)).argmax(dim=1).cpu().numpy()
    val_accuracy = accuracy_score(y_val.numpy(), val_pred)
    history.append({'epoch': epoch, 'train_loss': running_loss / len(Z_train),
                    'val_accuracy': val_accuracy})
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss: {history[-1]["train_loss"]:.4f} | val accuracy: {val_accuracy:.2%}')
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_weights = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print('Early stopping: validation accuracy did not improve')
            break

head.load_state_dict(best_weights)
head.eval()
print(f'Best validation accuracy: {best_accuracy:.2%}')
pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)

torch.save({'head_state_dict': best_weights, 'feature_mean': feature_mean,
            'feature_std': feature_std, 'class_to_idx': train_ds.class_to_idx,
            'backbone': MODEL_NAME}, OUTPUT_DIR / 'linear_head.pt')

## 8. Validate and inspect the baseline

In [ ]:
with torch.inference_mode():
    val_logits = head(Z_val.to(DEVICE))
    val_prediction = val_logits.argmax(dim=1).cpu().numpy()
print('Validation accuracy:', f'{accuracy_score(y_val.numpy(), val_prediction):.2%}')
print('Validation macro-F1:', f'{f1_score(y_val.numpy(), val_prediction, average="macro"):.4f}')

hist = pd.DataFrame(history)
plt.figure(figsize=(7, 3.6))
plt.plot(hist.epoch, hist.val_accuracy, marker='o', markersize=3)
plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.title('DINOv1 + linear head')
plt.grid(alpha=.25); plt.show()

## 9. Predict unlabeled test images and make Kaggle submission

There are no test labels in this notebook. Make sure your output has **exactly two columns**: `image_id,label`, matching the order of `sample_submission.csv`. Kaggle competition scoring should use integer class labels.

In [ ]:
with torch.inference_mode():
    test_logits = head(Z_test.to(DEVICE))
    test_predictions = test_logits.argmax(dim=1).cpu().numpy().astype(int)

submission = pd.DataFrame({'image_id': required_ids, 'label': test_predictions})
assert submission['label'].between(0, NUM_CLASSES-1).all()
assert len(submission) == len(test_files)
assert submission.image_id.is_unique
if sample_path.exists():
    original = pd.read_csv(sample_path, dtype={'image_id': str})
    assert submission['image_id'].tolist() == original['image_id'].tolist()

submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(submission_path, index=False)
print('Saved Kaggle submission:', submission_path.resolve())
print('Number of predictions:', len(submission))
display(submission.head(10))